# Semana 7 — NumPy II: estadística y filtrado

**Programación para Ciencia de Datos**

La semana pasada construiste arrays a partir de un CSV sucio y operaste con
ellos elemento a elemento. Esta semana cierras las dos herramientas que te
faltan para analizar datos de verdad:

1. **Estadística descriptiva vectorizada** — `median`, `std`, `percentile`, y
   cómo pedirlas por fila o por columna.
2. **Máscaras booleanas** — hacerle una pregunta a un array completo y quedarte
   solo con los valores que la cumplen.

Con eso se detectan **outliers**, que es el paso central de la **Práctica 3**.

> **Cómo usar este notebook.** Los ejemplos van sobre datos que no son los de tu
> práctica: latencias de un servidor y temperaturas de estaciones de monitoreo.
> Eso es deliberado. Aquí ves **cómo funciona cada herramienta**; aplicarla a tu
> dataset y armar el JSON que pide la P3 es tu trabajo, y está planteado en los
> ejercicios. No vas a encontrar aquí la práctica resuelta para copiar.

### Los datos de hoy

Dos juegos pequeños, a propósito: lo bastante chicos para verificar con los ojos
lo que calcula NumPy. Si no puedes comprobar a mano un resultado en 10 valores,
tampoco vas a notar que está mal en 100,000.

In [1]:
import time

import numpy as np

# Tiempos de respuesta de un servidor, en milisegundos.
# Fijate en el sexto valor: ese es el que va a dar de que hablar hoy.
ms = np.array([120.0, 95.0, 110.0, 105.0, 98.0, 2400.0, 115.0, 100.0, 92.0, 108.0])

print("latencias (ms):", ms)
print("cuantas        :", ms.size)

latencias (ms): [ 120.   95.  110.  105.   98. 2400.  115.  100.   92.  108.]
cuantas        : 10


In [2]:
# Cuatro estaciones de monitoreo (filas) x cinco dias (columnas), en grados C
temp = np.array([
    [21.5, 22.0, 20.8, 23.1, 22.4],
    [18.2, 17.9, 19.0, 18.5, 17.6],
    [27.3, 28.1, 26.9, 29.0, 28.5],
    [24.0, 23.5, 24.8, 25.1, 23.9],
])

print("shape:", temp.shape, "-> 4 estaciones, 5 dias")
print(temp)

shape: (4, 5) -> 4 estaciones, 5 dias
[[21.5 22.  20.8 23.1 22.4]
 [18.2 17.9 19.  18.5 17.6]
 [27.3 28.1 26.9 29.  28.5]
 [24.  23.5 24.8 25.1 23.9]]


## Sesión 1 — Estadística descriptiva vectorizada

La semana pasada viste `sum`, `mean`, `min` y `max` como adelanto. Hoy va la
batería completa, y con ella dos detalles que cambian el resultado numérico y
que casi nadie lee: el **denominador de la desviación estándar** y la
**interpolación de los percentiles**.

### La media y la mediana, en fórmula

La **media aritmética** suma todos los valores y divide entre cuántos son:

$$\bar{x} \;=\; \frac{1}{n}\sum_{i=1}^{n} x_i$$

La **mediana** es el valor que queda justo a la mitad una vez que ordenas los
datos. Con $x_{(1)} \le x_{(2)} \le \dots \le x_{(n)}$ los valores ordenados:

$$\text{mediana} \;=\;
\begin{cases}
x_{\left(\frac{n+1}{2}\right)} & \text{si } n \text{ es impar} \\[8pt]
\dfrac{x_{\left(\frac{n}{2}\right)} + x_{\left(\frac{n}{2}+1\right)}}{2}
  & \text{si } n \text{ es par}
\end{cases}$$

Fíjate en la diferencia: en la media **entra cada valor**, así que uno enorme
arrastra el resultado. En la mediana solo importa **el orden**, y un valor
enorme pesa igual que cualquier otro.

### `median`: el valor de en medio

Con las latencias se ve de golpe: un solo registro de 2,400 ms arrastra la media
a más del triple de lo que tarda una petición típica. La mediana ni se mueve.

In [3]:
print(ms)

[ 120.   95.  110.  105.   98. 2400.  115.  100.   92.  108.]


In [4]:
print("media   :", round(ms.mean(), 2))
print("mediana :", np.median(ms))
print()
print("ordenadas:", np.sort(ms))

media   : 334.3
mediana : 106.5

ordenadas: [  92.   95.   98.  100.  105.  108.  110.  115.  120. 2400.]


In [5]:
# La mediana, a mano con la formula. Son 10 valores (n par), asi que es el
# promedio de los dos de en medio. Ojo: en Python los indices empiezan en 0,
# asi que los de en medio son n//2 - 1 y n//2.
ordenadas = np.sort(ms)
n = ordenadas.size

if n % 2 == 1:
    mediana_a_mano = ordenadas[n // 2]
else:
    mediana_a_mano = (ordenadas[n // 2 - 1] + ordenadas[n // 2]) / 2

print("los dos de en medio:", ordenadas[n // 2 - 1], "y", ordenadas[n // 2])
print("a mano  :", mediana_a_mano)
print("np.median:", np.median(ms))
print("coinciden?", np.isclose(mediana_a_mano, np.median(ms)))

los dos de en medio: 105.0 y 108.0
a mano  : 106.5
np.median: 106.5
coinciden? True


La mediana de 106.5 describe el servidor; la media de 334.3, no. Por eso la P3
te pide **las dos**: juntas te dicen si tu columna tiene valores extremos, antes
de que te pongas a buscarlos.

> **Ojo:** `np.median` es una función, no un método. `ms.median()` no existe y te
> da `AttributeError`. Con `mean`, `std`, `min`, `max` y `sum` puedes usar
> cualquiera de las dos formas; con `median` y `percentile`, solo `np.algo(...)`.

In [6]:
try:
    ms.median()
except AttributeError as e:
    print("AttributeError:", e)

AttributeError: 'numpy.ndarray' object has no attribute 'median'


### `std`: la trampa del denominador

La **varianza** es el promedio de las distancias al cuadrado respecto a la media,
y la **desviación estándar** es su raíz cuadrada —lo que la devuelve a las
unidades originales, en nuestro caso milisegundos:

$$\sigma^2 \;=\; \frac{1}{n}\sum_{i=1}^{n}\left(x_i - \bar{x}\right)^2
\qquad\qquad
\sigma \;=\; \sqrt{\frac{1}{n}\sum_{i=1}^{n}\left(x_i - \bar{x}\right)^2}$$

Esa es la **poblacional**: divide entre $n$. Pero hay otra, la **muestral**, que
divide entre $n-1$ y se usa cuando tus datos son una muestra de algo más grande
y quieres estimar la desviación de ese todo:

$$s \;=\; \sqrt{\frac{1}{n-1}\sum_{i=1}^{n}\left(x_i - \bar{x}\right)^2}$$

NumPy las unifica en una sola fórmula con el parámetro `ddof` (*delta degrees of
freedom*: cuánto le restas a $n$):

$$\texttt{np.std(a, ddof=d)} \;=\;
\sqrt{\frac{1}{n-d}\sum_{i=1}^{n}\left(x_i - \bar{x}\right)^2}$$

Con `ddof=0` —**el valor por omisión**— te da la poblacional; con `ddof=1`, la
muestral. El numerador es idéntico en las dos; lo único que cambia es el
denominador, y por eso la muestral siempre sale un poco más grande.

In [7]:
a = np.array([2.0, 4.0, 4.0, 4.0, 5.0, 5.0, 7.0, 9.0])
print("media:", a.mean())
print()
print("np.std(a)        ->", np.std(a), " (poblacional, entre n)")
print("np.std(a, ddof=1)->", round(np.std(a, ddof=1), 4), " (muestral, entre n-1)")

media: 5.0

np.std(a)        -> 2.0  (poblacional, entre n)
np.std(a, ddof=1)-> 2.1381  (muestral, entre n-1)


In [8]:
# Lo mismo, a mano, para ver de donde sale cada numero
media = a.mean()
suma_cuadrados = ((a - media) ** 2).sum()

print("suma de cuadrados:", suma_cuadrados)
print("entre n   =", a.size, "->", (suma_cuadrados / a.size) ** 0.5)
print("entre n-1 =", a.size - 1, "->", round((suma_cuadrados / (a.size - 1)) ** 0.5, 4))

suma de cuadrados: 32.0
entre n   = 8 -> 2.0
entre n-1 = 7 -> 2.1381


**Para la P3 va la poblacional**, o sea `np.std(arr)` tal cual, sin `ddof`. Es
también la que calcula un bucle de Python puro que divide entre `n`, y por eso
el benchmark de las dos versiones tiene que dar el mismo número.

> **Guárdate este dato para la P4:** `pandas` hace lo contrario. El `.std()` de
> un `DataFrame` usa `ddof=1` por defecto. Si comparas el resultado de NumPy
> contra el de pandas sobre la misma columna y no cuadran en los decimales, casi
> siempre es esto y no un error tuyo.

### `percentile`: cortar la distribución

El percentil 25 es el valor por debajo del cual queda el 25% de los datos. Los
percentiles 25 y 75 son el *primer* y *tercer cuartil*, y junto con la mediana
—que es el percentil 50— describen la forma de la distribución.

La distancia entre esos dos cuartiles se llama **rango interpercentil** o IQR, y
mide la dispersión del 50% central de tus datos:

$$\text{IQR} \;=\; P_{75} - P_{25}$$

Es la alternativa robusta a la desviación estándar: como solo depende de dos
cortes de la distribución, un valor extremo no la mueve.

Puedes pedir varios percentiles de un jalón pasando una lista, y lo que regresa
es un array.

In [9]:
print(ms)

[ 120.   95.  110.  105.   98. 2400.  115.  100.   92.  108.]


In [10]:
cuartiles = np.percentile(ms, [25, 50, 75])

print("Medidas:", ms)
print("Ordenados:", np.sort(ms))
print("p25, p50, p75:", cuartiles)
print("tipo de lo que regresa:", type(cuartiles).__name__, cuartiles.shape)
print()
print("el p50 es la mediana:", np.percentile(ms, 50) == np.median(ms))

Medidas: [ 120.   95.  110.  105.   98. 2400.  115.  100.   92.  108.]
Ordenados: [  92.   95.   98.  100.  105.  108.  110.  115.  120. 2400.]
p25, p50, p75: [ 98.5  106.5  113.75]
tipo de lo que regresa: ndarray (3,)

el p50 es la mediana: True


### Un percentil casi nunca es un valor de tus datos

`np.percentile` **interpola**: si el corte cae entre dos observaciones, devuelve
un valor intermedio que no está en el array. Mira estos cuatro números:

In [11]:
b = np.array([10.0, 20.0, 30.0, 40.0])

print("datos:", b)
print("p25  :", np.percentile(b, 25), "<- no esta en los datos")
print("p50  :", np.percentile(b, 50))
print("p75  :", np.percentile(b, 75))

datos: [10. 20. 30. 40.]
p25  : 17.5 <- no esta en los datos
p50  : 25.0
p75  : 32.5


El p25 cae a tres cuartos del camino entre 10 y 20, y NumPy devuelve 17.5. Es el
comportamiento correcto y es lo que espera la P3; solo no te sorprendas cuando tu
`percentil_25` sea un número con decimales que no aparece en tu CSV.

### De dónde sale ese 17.5

El método que NumPy usa por omisión se llama *linear*. Con los datos ordenados
$x_0 \le x_1 \le \dots \le x_{n-1}$ —índices desde 0, como en Python— y un
percentil $p$ entre 0 y 100, primero se calcula una **posición ficticia**:

$$h \;=\; (n-1)\,\frac{p}{100}$$

Si $h$ cae exactamente en un índice entero, el percentil es ese valor. Si cae
entre dos, se interpola linealmente entre sus vecinos:

$$P_p \;=\; x_{\lfloor h \rfloor} \;+\;
\bigl(h - \lfloor h \rfloor\bigr)\,
\bigl(x_{\lfloor h \rfloor + 1} - x_{\lfloor h \rfloor}\bigr)$$

Con nuestros cuatro números: $n = 4$ y $p = 25$ dan $h = 3 \cdot 0.25 = 0.75$, o
sea entre $x_0 = 10$ y $x_1 = 20$, a tres cuartos del camino:
$10 + 0.75 \cdot (20 - 10) = 17.5$.

Nota de paso: con $p = 50$ la fórmula **es** la mediana. Por eso el p50 y la
mediana siempre coinciden, y no son dos cálculos distintos.

In [12]:
# La formula del percentil, a mano, contra np.percentile
def percentil_a_mano(arr, p):
    ordenadas = np.sort(arr)
    n = ordenadas.size
    h = (n - 1) * p / 100.0
    piso = int(np.floor(h))
    if piso >= n - 1:                  # el caso p = 100
        return float(ordenadas[-1])
    fraccion = h - piso
    return float(ordenadas[piso] + fraccion * (ordenadas[piso + 1] - ordenadas[piso]))


for p in (25, 50, 75):
    a_mano = percentil_a_mano(b, p)
    de_numpy = float(np.percentile(b, p))
    print("p%-3d a mano: %-6s numpy: %-6s coinciden: %s"
          % (p, a_mano, de_numpy, np.isclose(a_mano, de_numpy)))

p25  a mano: 17.5   numpy: 17.5   coinciden: True
p50  a mano: 25.0   numpy: 25.0   coinciden: True
p75  a mano: 32.5   numpy: 32.5   coinciden: True


In [ ]:
# Y sobre las latencias, que no son tan redondas
for p in (25, 50, 75):
    print("p%-3d a mano: %-8.3f numpy: %-8.3f" % (
        p, percentil_a_mano(ms, p), np.percentile(ms, p)))
print()
print("el p50 es la mediana:", np.isclose(percentil_a_mano(ms, 50), np.median(ms)))

### Operaciones por eje: `axis=0` y `axis=1`

Sobre una matriz, una agregación sin más te colapsa **todo** a un número. El
argumento `axis` dice en qué dirección colapsar, y es la parte que todo el mundo
confunde. La regla corta: **`axis` es la dimensión que desaparece.**

Con `temp`, de forma `(4, 5)`:

- `axis=0` recorre las filas y deja **una cifra por columna** → por día
- `axis=1` recorre las columnas y deja **una cifra por fila** → por estación

In [13]:
print("Mediciones de temperatura (C):\n\n", temp)

Mediciones de temperatura (C):

 [[21.5 22.  20.8 23.1 22.4]
 [18.2 17.9 19.  18.5 17.6]
 [27.3 28.1 26.9 29.  28.5]
 [24.  23.5 24.8 25.1 23.9]]


In [14]:
print("todo        :", round(temp.mean(), 2), "-> un solo numero")
print()
print("axis=0 (por dia)      :", np.round(temp.mean(axis=0), 2), "->", temp.mean(axis=0).shape)
print("axis=1 (por estacion) :", np.round(temp.mean(axis=1), 2), "->", temp.mean(axis=1).shape)

todo        : 23.1 -> un solo numero

axis=0 (por dia)      : [22.75 22.88 22.87 23.92 23.1 ] -> (5,)
axis=1 (por estacion) : [21.96 18.24 27.96 24.26] -> (4,)


In [15]:
# El truco para no equivocarse: mira que shape te quedo.
# (4, 5) con axis=0 -> se va el 4, queda (5,). Con axis=1 -> se va el 5, queda (4,).
print("shape original:", temp.shape)
print("axis=0 ->", temp.mean(axis=0).shape, " (se fue el 4: las estaciones)")
print("axis=1 ->", temp.mean(axis=1).shape, " (se fue el 5: los dias)")

shape original: (4, 5)
axis=0 -> (5,)  (se fue el 4: las estaciones)
axis=1 -> (4,)  (se fue el 5: los dias)


`axis` funciona igual en `median`, `std`, `min`, `max` y `percentile`. Esto te
deja sacar el resumen de varias columnas en una sola llamada:

In [19]:
resumen = np.percentile(temp, [25, 50, 75], axis=1)

print("filas = p25, p50, p75 | columnas = las 4 estaciones")
print(np.round(resumen, 2))
print()
print("Desviacion estandar (std) por estacion:", np.round(temp.std(axis=1), 2))
print("la estacion mas estable (menor std):", temp.std(axis=1).argmin())

filas = p25, p50, p75 | columnas = las 4 estaciones
[[21.5 17.9 27.3 23.9]
 [22.  18.2 28.1 24. ]
 [22.4 18.5 28.5 24.8]]

Desviacion estandar (std) por estacion: [0.78 0.48 0.77 0.6 ]
la estacion mas estable (menor std): 1


### `nan` otra vez: las versiones `nan*`

La semana pasada quedó claro que un `nan` contamina cualquier cálculo. Con la
estadística de hoy pasa lo mismo, y NumPy ofrece una versión `nan*` de cada
función que simplemente ignora los huecos.

In [20]:
con_hueco = np.array([120.0, 95.0, np.nan, 105.0, 98.0, np.nan])
print(con_hueco)
print(con_hueco.mean())

[120.  95.  nan 105.  98.  nan]
nan


In [21]:


print("median      :", np.median(con_hueco), "<- inservible")
print("nanmedian   :", np.nanmedian(con_hueco))
print()
print("std         :", np.std(con_hueco))
print("nanstd      :", round(np.nanstd(con_hueco), 2))
print()
print("percentile  :", np.percentile(con_hueco, [25, 75]))
print("nanpercentile:", np.nanpercentile(con_hueco, [25, 75]))

median      : nan <- inservible
nanmedian   : 101.5

std         : nan
nanstd      : 9.66

percentile  : [nan nan]
nanpercentile: [ 97.25 108.75]


In [23]:
print(con_hueco)
print(np.isnan(con_hueco))


[120.  95.  nan 105.  98.  nan]
[False False  True False False  True]


Tienes entonces **dos rutas** para trabajar una columna con huecos, y las dos son
válidas en la P3:

| Ruta | Cómo | Qué cuidar |
|---|---|---|
| Filtrar primero | `validos = col[~np.isnan(col)]` y después la estadística normal | `validos.size` es tu `n_valores_validos` |
| Usar `nan*` | `np.nanmean(col)`, `np.nanmedian(col)`, ... sobre el array completo | `col.size` **no** es tu `n_valores_validos`; hay que contar aparte |

La primera es menos propensa a errores porque el `n` que reportas sale del mismo
array con el que calculaste. Elige una y sé consistente: mezclarlas es cómo
acaba uno reportando un `n` que no corresponde a la media de al lado.

### Ejercicios sesión 1

Ninguno está resuelto. Los primeros son de calentamiento; del 4 en adelante ya
son sobre **tu** dataset.

In [ ]:
# Ejercicio 1: Calcula la mediana de cada DIA en `temp` (una cifra por dia).
# Imprime tambien que dia fue el mas caluroso segun la mediana.

# Tu codigo aqui:

In [ ]:
# Ejercicio 2: Para el array `ms`, imprime media, mediana, std poblacional,
# std muestral, minimo, maximo, p25 y p75, todos redondeados a 2 decimales.
# Son los ocho estadisticos que la P3 pide por columna.

# Tu codigo aqui:

In [ ]:
# Ejercicio 3: El rango interpercentil (IQR) es p75 - p25. Calculalo para
# `ms` y para cada estacion de `temp`. Que te dice un IQR chico?

# Tu codigo aqui:

In [ ]:
# Ejercicio 4: Carga UNA de tus dos columnas numericas desde
# datos/{tema}-ruido.csv (el patron de la semana 6: open, split("|"),
# localizar por nombre con .index(), convertir marcando lo invalido) y
# reporta los ocho estadisticos del ejercicio 2 sobre los valores validos.
# Cuantos valores validos te quedaron?

# Tu codigo aqui:

In [ ]:
# Ejercicio 5: Repite el ejercicio 4 con tu segunda columna numerica, pero
# ahora SIN filtrar: usa las funciones nan* sobre el array completo.
# Verifica con np.allclose que las dos rutas dan lo mismo.

# Tu codigo aqui:

## Sesión 2 — Máscaras booleanas y filtrado

Una **máscara booleana** es un array de `True`/`False` del mismo tamaño que tus
datos, donde cada posición responde una pregunta sobre el valor correspondiente.
Es la forma en que NumPy reemplaza al `if` dentro de un `for`.

### Una comparación produce un array, no un `True` suelto

In [ ]:
lentas = ms > 150

print("datos   :", ms)
print("ms > 150:", lentas)
print("dtype   :", lentas.dtype)

### Contar y medir con la máscara

Como `True` vale 1 y `False` vale 0, sobre la máscara puedes usar la misma
aritmética de siempre: `sum()` cuenta cuántas cumplen y `mean()` te da
directamente la **proporción**.

En notación matemática, la máscara es la *función indicadora* de tu condición
—vale 1 cuando se cumple y 0 cuando no—, así que sumarla cuenta y promediarla da
la proporción:

$$\text{cuántas} \;=\; \sum_{i=1}^{n} \mathbf{1}\bigl[\,\text{cond}(x_i)\,\bigr]
\qquad\qquad
\text{proporción} \;=\; \frac{1}{n}\sum_{i=1}^{n}
  \mathbf{1}\bigl[\,\text{cond}(x_i)\,\bigr]$$

Y el porcentaje es la proporción por 100. Esa segunda fórmula es exactamente lo
que hace `mascara.mean()`, y es la razón de que promediar booleanos tenga
sentido.

In [ ]:
print("cuantas cumplen :", lentas.sum())
print("proporcion      :", lentas.mean())
print("porcentaje      :", round(100 * lentas.mean(), 2), "%")
print()
print("alguna cumple?  :", lentas.any())
print("todas cumplen?  :", lentas.all())

Ese `round(100 * mascara.mean(), 2)` es exactamente la forma de calcular el
campo `porcentaje` que la P3 pide dentro de sus bloques de outliers. Lo único
que tendrás que decidir es **sobre qué total** lo calculas.

### Filtrar de verdad: la máscara como índice

Pasar la máscara entre corchetes devuelve **solo los valores donde era `True`**.
El array que sale es más chico, y eso es la señal de que filtraste de verdad.

In [ ]:
print("las lentas       :", ms[lentas])
print("shape original   :", ms.shape)
print("shape filtrado   :", ms[lentas].shape)
print()
print("max de las lentas:", ms[lentas].max())
print("cuantas rapidas  :", ms[~lentas].size, "<- el ~ niega la mascara")

> **El indexado booleano devuelve una copia, no una vista.** Es lo contrario del
> *slicing* de la semana pasada. Modificar el resultado no toca el original:

In [ ]:
sub = ms[ms > 150]
sub[0] = 0.0

print("sub:", sub)
print("ms sigue intacto, su maximo es:", ms.max())

### Combinar condiciones: `&`, `|`, `~`

Para juntar preguntas se usan los operadores de bits, **no** las palabras de
Python:

| Quiero | Operador |
|---|---|
| las dos cosas | `&` |
| una u otra | `\|` |
| lo contrario | `~` |

Y cada condición va **entre paréntesis**, siempre.

In [ ]:
normales = (ms >= 95) & (ms <= 115)

print("entre 95 y 115   :", normales)
print("cuantas          :", normales.sum())
print("sus valores      :", ms[normales])
print()
print("las que NO       :", ms[~normales])

### Dos trampas que te van a pasar hoy

**Primera: sin paréntesis no compila.** `&` tiene más precedencia que `>=`, así
que Python agrupa mal y truena:

In [ ]:
try:
    ms >= 95 & ms <= 115
except TypeError as e:
    print("TypeError:", e)
    print()
    print("Python leyo:  ms >= (95 & ms) <= 115")

**Segunda: `and` y `or` no sirven para esto.** Python intenta evaluar el array
completo como un solo `True`/`False` y no puede decidir:

In [ ]:
try:
    (ms > 100) and (ms < 200)
except ValueError as e:
    print("ValueError:", e)

Es probablemente el error más frecuente de la semana. Si lo ves, busca un `and`
o un `or` en tu condición y cámbialo por `&` o `|`.

### Preguntar por lo que queda *fuera* de un rango

Hasta ahora preguntamos por lo que cae **dentro**. Lo de fuera es la unión de dos
condiciones, una por cada lado, y funciona igual sobre una matriz: la máscara
tiene la forma de `temp`, pero el filtrado devuelve los valores en 1-D.

In [ ]:
# Rango operativo de los sensores: entre 19 y 27 grados
fuera = (temp < 19.0) | (temp > 27.0)

print("la mascara tiene la forma de temp:", fuera.shape)
print(fuera)
print()
print("lecturas fuera de rango:", temp[fuera], "<- en 1-D")
print("cuantas                :", fuera.sum(), "de", temp.size)
print("por estacion           :", fuera.sum(axis=1), "<- axis tambien aqui")

Guarda bien esta forma. El paso de outliers de la P3 es esta misma pregunta, solo
que los dos límites no te los dan: los calculas tú a partir de los datos. Eso es
la sesión 3.

### `np.where`: decidir valor por valor

Con tres argumentos, `np.where(condicion, si_cumple, si_no)` es un `if/else`
vectorizado: construye un array nuevo eligiendo de un lado o del otro.

In [ ]:
# Topar los valores extremos en 150 (sin perder registros)
acotado = np.where(ms > 150, 150.0, ms)

print("original:", ms)
print("acotado :", acotado)
print()
# Etiquetar en lugar de filtrar
etiqueta = np.where(ms > 150, "lenta", "normal")
print("etiquetas:", etiqueta)

Con **un solo** argumento, `np.where` cambia de oficio y te dice **en qué
posiciones** se cumple la condición. Devuelve una tupla (una entrada por
dimensión), de ahí el `[0]` en el caso 1-D.

In [ ]:
posiciones = np.where(ms > 150)[0]

print("posiciones:", posiciones)
print("sus valores:", ms[posiciones])
print()
# En 2-D son dos arrays: filas y columnas
filas, columnas = np.where(temp > 28.0)
print("temp > 28 en (fila, columna):", list(zip(filas, columnas)))

Eso es útil cuando necesitas **volver a la fila original** del CSV: la posición
del array es la posición de la fila, siempre que no hayas descartado valores
sobre la marcha.

### Ejercicios sesión 2

In [ ]:
# Ejercicio 6: Cuantas lecturas de `temp` estan por encima del promedio de
# SU PROPIA estacion? (pista: temp.mean(axis=1) te da un promedio por fila;
# necesitas reshape(-1, 1) para que el broadcasting lo compare bien).

# Tu codigo aqui:

In [ ]:
# Ejercicio 7: Usando `ms`, construye la mascara de las latencias que estan
# entre el p25 y el p75. Cuantas son? Que porcentaje del total?

# Tu codigo aqui:

In [ ]:
# Ejercicio 8: Con np.where, crea un array que diga "frio" por debajo de 20,
# "templado" entre 20 y 26, y "caliente" por encima de 26, para `temp`.
# (pista: se pueden anidar dos np.where).

# Tu codigo aqui:

In [ ]:
# Ejercicio 9: En tu columna numerica_1, cuantos valores son negativos?
# Que porcentaje de los validos? Imprime el minimo y el maximo de ESOS
# valores negativos. (Los negativos son ruido puesto a proposito en tu
# dataset; en la P2 varios equipos los descartaron por error.)

# Tu codigo aqui:

In [ ]:
# Ejercicio 10: Escribe una funcion
#   fuera_de_rango(arr, limite_inf, limite_sup)
# que devuelva la mascara de los valores estrictamente fuera del rango.
# Pruebala con `ms` y con tu columna. La vas a necesitar el viernes.

# Tu codigo aqui:

## Sesión 3 — Outliers y cómo medir tiempos

Ya tienes las dos piezas: estadística para describir una columna, y máscaras para
quedarte con parte de ella. Juntarlas da la detección de outliers.

### La regla de las k desviaciones

Un **outlier** es un valor tan alejado del centro que vale la pena mirarlo
aparte. La regla más común lo define por distancia a la media, medida **en
desviaciones estándar**. Los dos límites son:

$$L_{\text{inf}} \;=\; \bar{x} - k\,\sigma
\qquad\qquad
L_{\text{sup}} \;=\; \bar{x} + k\,\sigma$$

y es outlier todo valor que quede **estrictamente fuera** del intervalo
$[\,L_{\text{inf}},\, L_{\text{sup}}\,]$:

$$x_i \text{ es outlier} \;\iff\;
x_i < \bar{x} - k\sigma \;\;\text{o}\;\; x_i > \bar{x} + k\sigma$$

Las dos condiciones se pueden escribir como una sola, con valor absoluto —es la
misma regla, y a veces es más cómoda de programar:

$$x_i \text{ es outlier} \;\iff\; \left|\,x_i - \bar{x}\,\right| > k\,\sigma$$

Esa cantidad, la distancia a la media medida en desviaciones, tiene nombre
propio: el **puntaje z**.

$$z_i \;=\; \frac{x_i - \bar{x}}{\sigma}
\qquad\Longrightarrow\qquad
x_i \text{ es outlier} \;\iff\; |z_i| > k$$

Dicho así, la regla es: *es atípico lo que está a más de k desviaciones de la
media*. Y el porcentaje que se reporta es el conteo sobre el total de válidos:

$$\text{porcentaje} \;=\; 100 \cdot \frac{\#\{\text{outliers}\}}{n}$$

`k = 3` es la convención, y es la que usa la P3. En datos con forma de campana
deja fuera alrededor del 0.3% de las observaciones, así que lo que marque es
poco y vale la pena revisarlo.

In [ ]:
media = ms.mean()
std = ms.std()

print("media:", round(media, 2))
print("std  :", round(std, 2))
print()
for k in (1, 2, 3):
    inferior = media - k * std
    superior = media + k * std
    marcados = ms[(ms < inferior) | (ms > superior)]
    print("k =", k, "-> limites [", round(inferior, 2), ",", round(superior, 2), "]",
          "-> marca", marcados)

### Lo que acaba de pasar, y es la lección de hoy

Con `k = 3` **no se marcó nada**, aunque el 2,400 esté a la vista. El límite
superior quedó en 2,400.15 y el valor es 2,400.0: se escapó por 15 centésimas.

La razón es que **el outlier infla la desviación estándar que debería
detectarlo**. Con solo 10 valores, uno extremo levanta la `std` de ~10 a 688, y
los límites se abren tanto que ya lo abarcan. Es el punto débil conocido de este
método: no es robusto cuando hay pocos datos o cuando el valor extremo es muy
extremo.

Dos consecuencias prácticas:

1. **En tu P3 esto no te va a pasar igual**, porque trabajas con ~99,000 valores:
   un puñado de extremos ya no mueve la `std` del conjunto, y la regla sí marca.
   Pero no supongas que marcará mucho; en varios temas el porcentaje es menor al
   1%, y eso está bien.
2. **Un `n_outliers` de cero es un resultado posible**, no necesariamente un bug.
   Lo que sí es un bug es que te salga `nan`, o que marques la mitad de tus datos.

> Métodos más robustos —como el del IQR, que usa `p25` y `p75` en vez de media y
> desviación— no sufren esto, porque los cuartiles no se mueven por un valor
> extremo. En la P6 vas a usar precisamente ese, cuando dibujes diagramas de
> caja.

### Lo mismo, visto con puntajes z

Si en vez de los límites miras el **puntaje z** de cada valor, lo que pasó se lee
de un golpe. Y de paso se comprueba que las tres formas de escribir la regla
—por límites, por valor absoluto y por z— producen exactamente la misma máscara.

In [ ]:
z = (ms - media) / std

print("puntajes z:", np.round(z, 3))
print()
print("el z del valor extremo:", z.max())
print("se pasa de 3?", z.max() > 3)

Ahí está el culpable: el z del 2,400 es **2.9998**, no 3.0001. Por eso `k = 3`
lo deja pasar y `k = 2` sí lo marca. En pantalla, redondeado, parece un 3
exacto; la decisión se juega en el cuarto decimal.

Y las tres formulaciones son equivalentes:

In [ ]:
K = 2

por_limites = (ms < media - K * std) | (ms > media + K * std)
por_absoluto = np.abs(ms - media) > K * std
por_z = np.abs(z) > K

print("por limites :", por_limites)
print("por absoluto:", por_absoluto)
print("por z       :", por_z)
print()
print("las tres coinciden?", np.array_equal(por_limites, por_absoluto)
      and np.array_equal(por_limites, por_z))

### Un detalle que sí cambia tu resultado: redondear al final, no antes

La P3 te pide reportar la media y la desviación redondeadas a 2 decimales. Si
calculas los límites **a partir de los valores ya redondeados**, arrastras el
error multiplicado por 3.

In [ ]:
# Con precision completa
limite_exacto = ms.mean() - 3 * ms.std()

# Con los valores ya redondeados, como quedarian en el reporte
media_r = round(float(ms.mean()), 2)
std_r = round(float(ms.std()), 2)
limite_redondeado = media_r - 3 * std_r

print("limite con precision completa:", limite_exacto)
print("limite desde los redondeados :", limite_redondeado)
print("diferencia                   :", round(abs(limite_exacto - limite_redondeado), 6))

Son centésimas, y con 10 valores no cambia nada. Con 99,000 valores, un límite
desplazado unas centésimas puede dejar dentro o fuera a los que caen justo en la
frontera, y tu `n_outliers` cambia. **Calcula con los números completos y redondea
solo al escribir el JSON.**

### Medir tiempos sin engañarte

La P3 te pide comparar un bucle de Python contra NumPy. El **factor de
velocidad** es simplemente el cociente de los dos tiempos:

$$\text{factor} \;=\; \frac{t_{\text{bucle}}}{t_{\text{NumPy}}}$$

Un factor de 200 quiere decir "NumPy tardó 200 veces menos". Fíjate en que el
denominador es el tiempo de NumPy: si sale `0.0`, la división es imposible, y de
ahí viene el problema del reloj que vemos en la regla 3.

Medir mal es fácil, así que aquí van las tres reglas, con el ejemplo sobre una
operación distinta a la que te toca medir: **la suma de los cuadrados**,
$\sum_i x_i^2$.

In [ ]:
rng = np.random.default_rng(7)             # semilla fija: reproducible
grande_np = rng.normal(500, 120, 200_000)
grande_py = grande_np.tolist()             # la misma data, como lista de Python

print("cuantos valores:", len(grande_py))

In [ ]:
# --- Bucle de Python puro ---
inicio = time.perf_counter()
total_py = 0.0
for v in grande_py:
    total_py += v * v
t_py = time.perf_counter() - inicio

# --- NumPy, vectorizado ---
inicio = time.perf_counter()
total_np = float((grande_np ** 2).sum())
t_np = time.perf_counter() - inicio

print("bucle:", round(total_py, 2), "en", round(t_py, 6), "s")
print("numpy:", round(total_np, 2), "en", round(t_np, 6), "s")
print()
print("calculan lo mismo?", np.allclose(total_py, total_np))
print("factor de velocidad:", round(t_py / t_np, 1), "x")

**Regla 1: comprueba que las dos versiones calculan lo mismo.** Un benchmark
donde cada lado hace algo distinto no mide velocidad, mide tu descuido. El
`np.allclose` de arriba no es decorativo.

**Regla 2: no midas la lectura del archivo.** El reloj arranca cuando ya tienes
los datos en memoria. Si metes el `open()` y el `split("|")` dentro de la
medición, el tiempo que domina es el disco y el factor que reportes será
absurdamente bajo.

**Regla 3: usa `time.perf_counter()`, no `time.time()`.** En Windows,
`time.time()` avanza a saltos de unos 16 milisegundos. Una operación vectorizada
sobre 100,000 valores tarda menos que eso, así que las dos lecturas del reloj
salen idénticas, la resta da exactamente `0.0`, y al dividir te truena con
`ZeroDivisionError`:

In [ ]:
inicio = time.time()
(grande_np ** 2).sum()
t_malo = time.time() - inicio

print("medido con time.time()        :", t_malo, "s")
print("medido con time.perf_counter():", t_np, "s")
print()
try:
    print("factor con time.time():", t_py / t_malo)
except ZeroDivisionError as e:
    print("ZeroDivisionError:", e, "<- el reloj no alcanza a ver nada")

`perf_counter` se usa igual que `time.time()` —arranca, termina, restas— pero
está hecho para medir intervalos y tiene resolución de nanosegundos.

Si de todas formas te sale un tiempo de `0.0`, la otra salida es **repetir la
operación** y dividir el total entre el número de repeticiones:

In [ ]:
REPETICIONES = 200

inicio = time.perf_counter()
for _ in range(REPETICIONES):
    (grande_np ** 2).sum()
t_total = time.perf_counter() - inicio

print("total de", REPETICIONES, "repeticiones:", round(t_total, 6), "s")
print("por repeticion               :", t_total / REPETICIONES, "s")

### Ejercicios sesión 3

Del 11 al 16 construyes, pieza por pieza, lo que entregas en la P3. Ninguno está
resuelto aquí: son tu práctica.

In [ ]:
# Ejercicio 11: Escribe una funcion
#   limites_k_sigma(arr, k=3)
# que devuelva la tupla (limite_inferior, limite_superior) sin redondear.
# Pruebala con `ms` y con una de tus columnas.

# Tu codigo aqui:

In [ ]:
# Ejercicio 12: Usando la funcion del ejercicio 11 y la del ejercicio 10,
# cuenta los outliers de tu columna numerica_1 e imprime: cuantos son, que
# porcentaje de los validos representan, y el minimo y el maximo de esos
# valores atipicos.

# Tu codigo aqui:

In [ ]:
# Ejercicio 13: Que pasa si tu columna NO tiene ningun outlier? Prueba tu
# codigo del ejercicio 12 con un array sin valores extremos, por ejemplo
# np.array([1.0, 2.0, 3.0, 4.0, 5.0]). Truena? Que deberia reportar en el
# minimo y el maximo de los outliers cuando no hay ninguno?

# Tu codigo aqui:

In [ ]:
# Ejercicio 14: Arma el benchmark que pide la P3: media y desviacion estandar
# con un bucle de Python puro contra np.mean y np.std, sobre la lista de
# valores validos de tu numerica_1 que ya cargaste. Verifica con allclose que
# los dos dan lo mismo (cuidado con el denominador de la std) y reporta el
# factor de velocidad.

# Tu codigo aqui:

In [ ]:
# Ejercicio 15: Escribe una funcion
#   estadisticas(arr)
# que devuelva un diccionario con las ocho claves que la P3 pide por columna,
# ya redondeadas a 2 decimales. Ojo: json no sabe escribir tipos de NumPy;
# convierte con float() o int() antes de guardar.
# Imprimela con json.dumps(..., indent=2, ensure_ascii=False).

# Tu codigo aqui:

In [ ]:
# Ejercicio 16 (reto): Junta todo en un solo script que lea tu CSV, procese
# tus DOS columnas numericas y escriba el JSON completo de la P3. Compara el
# n_valores_validos de cada columna contra el que reportaste en la P2: deben
# coincidir. Si no, una de las dos practicas tiene un error de filtrado.

# Tu codigo aqui:

---

## Formulario de la semana

Todo lo que calculaste hoy, en una sola hoja. Con $x_1, \dots, x_n$ los valores
**válidos** de una columna y $x_{(1)} \le \dots \le x_{(n)}$ los mismos ya
ordenados.

**Centro**

$$\bar{x} = \frac{1}{n}\sum_{i=1}^{n} x_i
\qquad\qquad
\text{mediana} =
\begin{cases}
x_{\left(\frac{n+1}{2}\right)} & n \text{ impar} \\[6pt]
\dfrac{x_{\left(\frac{n}{2}\right)} + x_{\left(\frac{n}{2}+1\right)}}{2} & n \text{ par}
\end{cases}$$

**Dispersión** — `ddof=0` es la poblacional y es la que pide la P3

$$\sigma^2 = \frac{1}{n}\sum_{i=1}^{n}(x_i-\bar{x})^2
\qquad
\sigma = \sqrt{\sigma^2}
\qquad
s = \sqrt{\frac{1}{n-1}\sum_{i=1}^{n}(x_i-\bar{x})^2}$$

**Percentiles** — método *linear*, índices desde 0

$$h = (n-1)\frac{p}{100}
\qquad
P_p = x_{\lfloor h\rfloor} + (h-\lfloor h\rfloor)\left(x_{\lfloor h\rfloor+1}-x_{\lfloor h\rfloor}\right)
\qquad
\text{IQR} = P_{75}-P_{25}$$

**Máscaras**

$$\text{cuántas} = \sum_{i=1}^{n}\mathbf{1}\bigl[\text{cond}(x_i)\bigr]
\qquad
\text{porcentaje} = \frac{100}{n}\sum_{i=1}^{n}\mathbf{1}\bigl[\text{cond}(x_i)\bigr]$$

**Outliers a k desviaciones** — con $k = 3$

$$L_{\text{inf}} = \bar{x}-k\sigma
\qquad
L_{\text{sup}} = \bar{x}+k\sigma
\qquad
\text{outlier} \iff |x_i-\bar{x}| > k\sigma \iff |z_i| > k$$

donde $z_i = \dfrac{x_i-\bar{x}}{\sigma}$.

**Benchmark**

$$\text{factor} = \frac{t_{\text{bucle}}}{t_{\text{NumPy}}}$$

> **Las dos que más cuestan puntos:** usar $n-1$ en la desviación cuando se pide
> $n$, y calcular $L_{\text{inf}}$ y $L_{\text{sup}}$ con $\bar{x}$ y $\sigma$ ya
> redondeados a 2 decimales en lugar de los completos.

---

## Resumen de la semana

**Estadística** — todas aceptan `axis`

| Qué quieres | Cómo |
|---|---|
| Media, mínimo, máximo | `a.mean()`, `a.min()`, `a.max()` |
| Mediana | `np.median(a)` — no existe `a.median()` |
| Desviación **poblacional** (la de la P3) | `np.std(a)` |
| Desviación muestral | `np.std(a, ddof=1)` |
| Percentiles, varios de un jalón | `np.percentile(a, [25, 50, 75])` |
| Ignorando huecos | `np.nanmean`, `np.nanmedian`, `np.nanstd`, `np.nanpercentile` |
| Por columna / por fila | `axis=0` / `axis=1` — **`axis` es la dimensión que desaparece** |

**Máscaras booleanas**

| Qué | Cómo |
|---|---|
| Preguntar | `a > 100` → array de `True`/`False` |
| Contar | `mascara.sum()` |
| Porcentaje | `round(100 * mascara.mean(), 2)` |
| Filtrar | `a[mascara]` → **copia**, de menor tamaño |
| Combinar | `(a > 10) & (a < 20)`, `\|`, `~` — **siempre con paréntesis** |
| Fuera de un rango | `(a < inferior) \| (a > superior)` |
| Elegir valor por valor | `np.where(cond, si, no)` |
| Dónde se cumple | `np.where(cond)[0]` |

**Outliers a k desviaciones**

Límites en `media ± k * std`, con `k = 3`, calculados con los valores **sin
redondear**. Es outlier lo que queda estrictamente fuera.

## Las cinco trampas de esta semana

1. **Usar `and` u `or` entre dos máscaras.** Da
   `ValueError: truth value of an array ... is ambiguous`. Van `&` y `|`.
2. **Olvidar los paréntesis.** `a > 10 & a < 20` no es lo que escribiste: `&`
   tiene más precedencia que las comparaciones.
3. **Confundir `axis=0` con `axis=1`.** Mira el `shape` que te quedó: la
   dimensión que desapareció es la que pusiste en `axis`.
4. **Reportar la desviación muestral.** `np.std()` ya es la poblacional, que es
   la que pide la P3; `ddof=1` es otra cosa. Y en pandas el default es el
   contrario.
5. **Calcular los límites de outliers con la media y la desviación ya
   redondeadas.** Redondea solo al escribir el JSON.

## Qué sigue

**La Práctica 3** (`labs/P3/`), que es aplicar lo de estas dos semanas a tus dos
columnas numéricas y entregar un JSON. Todo lo que necesitas está en la semana 6
(cargar una columna sucia a un array) y en esta; el armado es tuyo, y es a
propósito: los ejercicios 11 a 16 son el camino.

Dos cosas que conviene revisar antes de darla por terminada:

- tu `n_valores_validos` de cada columna debería **coincidir con el de tu P2**,
  porque es el mismo criterio sobre el mismo archivo
- tu `benchmark.n_filas` tiene que ser igual a `n_valores_validos` de tu
  `numerica_1`

**Después, la semana 8: Regex ligera**, y de ahí a **pandas**, donde todo esto se
escribe más corto. Pero el orden importa: quien entiende qué es una máscara
booleana usa pandas con criterio; quien no, copia recetas.